# Notebook 2 — Nettoyage et transformation des données

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    DateType,
    DoubleType,
    IntegerType,
)

spark = (
    SparkSession.builder
    .appName("TradeCorp - Nettoyage")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

DATA_PATH = "/home/jovyan/data"
TMP_PATH = "/home/jovyan/data/tmp"

def lire_csv(nom_fichier):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{DATA_PATH}/{nom_fichier}")
    )

df_customers = lire_csv("customers.csv")
df_orders = lire_csv("orders.csv")
df_order_details = lire_csv("order_details.csv")
df_products = lire_csv("products.csv")
df_categories = lire_csv("categories.csv")
df_suppliers = lire_csv("suppliers.csv")
df_employees = lire_csv("employees.csv")
df_shippers = lire_csv("shippers.csv")

dataframes = {
    "customers": df_customers,
    "orders": df_orders,
    "order_details": df_order_details,
    "products": df_products,
    "categories": df_categories,
    "suppliers": df_suppliers,
    "employees": df_employees,
    "shippers": df_shippers,
}

print("DataFrames chargés :", len(dataframes))

## Q11 — Comptage des valeurs nulles

In [ ]:
for nom, df in dataframes.items():
    print(f"\n===== VALEURS NULLES : {nom} =====")

    resultats = []

    for colonne in df.columns:
        nombre_nulls = (
            df.filter(F.col(colonne).isNull())
            .count()
        )

        resultats.append((colonne, nombre_nulls))

    df_nulls = spark.createDataFrame(
        resultats,
        ["colonne", "valeurs_nulles"]
    )

    df_nulls.show(len(resultats), truncate=False)

## Q12 — Traitement des valeurs nulles

In [ ]:
nombre_orders_avant = df_orders.count()

df_orders_clean = df_orders.dropna(
    subset=["shipped_date"]
)

nombre_orders_apres = df_orders_clean.count()

print("Commandes avant nettoyage :", nombre_orders_avant)
print("Commandes après nettoyage :", nombre_orders_apres)
print(
    "Commandes supprimées :",
    nombre_orders_avant - nombre_orders_apres
)

In [ ]:
medianes = df_products.approxQuantile(
    "unit_price",
    [0.5],
    0.01
)

mediane_unit_price = medianes[0] if medianes else 0.0

df_products_clean = df_products.fillna(
    {"unit_price": mediane_unit_price}
)

print("Médiane de unit_price :", mediane_unit_price)

df_products_clean.filter(
    F.col("unit_price").isNull()
).show()

## Q13 — Conversion des types

In [ ]:
colonnes_dates_orders = [
    "order_date",
    "required_date",
    "shipped_date",
]

for colonne in colonnes_dates_orders:
    df_orders_clean = df_orders_clean.withColumn(
        colonne,
        F.to_date(F.col(colonne), "yyyy-MM-dd")
    )

df_orders_clean.printSchema()

In [ ]:
df_order_details_clean = (
    df_order_details
    .withColumn(
        "unit_price",
        F.col("unit_price").cast(DoubleType())
    )
    .withColumn(
        "quantity",
        F.col("quantity").cast(IntegerType())
    )
    .withColumn(
        "discount",
        F.col("discount").cast(DoubleType())
    )
)

df_order_details_clean.printSchema()

## Q14 — Nettoyage des chaînes de caractères

In [ ]:
df_customers_clean = df_customers

colonnes_texte_customers = [
    champ.name
    for champ in df_customers.schema.fields
    if isinstance(champ.dataType, StringType)
]

for colonne in colonnes_texte_customers:
    df_customers_clean = df_customers_clean.withColumn(
        colonne,
        F.trim(F.col(colonne))
    )

df_customers_clean = (
    df_customers_clean
    .withColumn(
        "contact_name",
        F.initcap(F.col("contact_name"))
    )
    .withColumn(
        "country",
        F.upper(F.col("country"))
    )
)

df_customers_clean.select(
    "customer_id",
    "contact_name",
    "country"
).show(10, truncate=False)

## Q15 — Renommage des colonnes

In [ ]:
df_order_details_clean = (
    df_order_details_clean
    .withColumnRenamed("unit_price", "prix_unitaire")
    .withColumnRenamed("quantity", "quantite")
)

df_orders_clean = df_orders_clean.withColumnRenamed(
    "ship_via",
    "shipper_id"
)

print("Colonnes order_details :")
print(df_order_details_clean.columns)

print("\nColonnes orders :")
print(df_orders_clean.columns)

## Q16 — Création de la colonne sous_total

In [ ]:
df_order_details_clean = df_order_details_clean.withColumn("sous_total",F.round(F.col("prix_unitaire")*F.col("quantite")*(F.lit(1.0)-F.col("discount")),2))
df_order_details_clean.select("order_id",
                              "product_id",
                              "prix_unitaire",
                              "quantite",
                              "discount",
                              "sous_total"
                             ).show(10,truncate=False)

## Q17 - Création des colonnes conditionnelles

In [ ]:
# Produits en stock
df_products_clean = df_products_clean.withColumn(
    "en_stock",
    F.col("units_in_stock")>0
)
df_products_clean.select(
    "product_id",
    "product_name",
    "units_in_stock",
    "en_stock"
).show(10,truncate=False)

In [ ]:
# Commandes expédiées
df_orders_clean = df_orders_clean.withColumn(
    "is_shipped",
    F.col("shipped_date").isNotNull()
)
df_orders_clean.select(
    "order_id",
    "shipped_date",
    "is_shipped"
).show(10, truncate=False)

### Observation

La colonne `is_shipped` vaut toujours `True` dans le DataFrame nettoyé,
car les commandes dont `shipped_date` était nulle ont été supprimées à la
question Q12.

## Q18 — Recherche et suppression des doublons

In [ ]:
nombre_total_customers = df_customers_clean.count()

In [ ]:
nombre_customers_distincts = (
    df_customers_clean.select("customer_id").distinct().count())

In [ ]:
nombre_doublons = (nombre_total_customers - nombre_customers_distincts)

In [ ]:
print("Nombre total de clients :", nombre_total_customers)
print(
    "Nombre d'identifiants distincts :",
    nombre_customers_distincts
)
print("Nombre de doublons :", nombre_doublons)

In [ ]:
df_customers_clean = (
    df_customers_clean
    .dropDuplicates(["customer_id"])
)

print(
    "Nombre de clients après suppression :",
    df_customers_clean.count()
)

## Q19 — Filtrage des commandes et des produits

In [ ]:
df_orders_1997 = df_orders_clean.filter(F.year(F.col("order_date"))==1997)
print("Nombre de commandes de 1997 :", df_orders_1997.count())
df_orders_1997.select("order_id", "order_date").show(10)

In [ ]:
# Produits actifs et disponibles
df_products_actifs = df_products_clean.filter((F.col("units_in_stock")>0) & (F.col("discontinued").cast("int")==0))
print("Nombre de produits actifs et en stock : ", df_products_actifs.count())
df_products_actifs.select("product_id", "product_name","units_in_stock","discontinued").show(10, truncate=False)

## Q20 — Sélection des colonnes employés

In [ ]:
df_employees_clean = (df_employees.select("employee_id","first_name","last_name","title","hire_date","city","country",).withColumn("hire_date",F.to_date(F.col("hire_date"),"yyyy-MM-dd")).withColumn("full_name",F.concat_ws(" ",F.col("first_name"),F.col("last_name"))))
df_employees_clean.show(truncate=False)
df_employees_clean.printSchema()

## Écriture des DataFrames nettoyés en Parquet

In [ ]:
dataframes_nettoyes = {
    "customers": df_customers_clean,
    "orders": df_orders_clean,
    "order_details": df_order_details_clean,
    "products": df_products_clean,
    "categories": df_categories,
    "suppliers": df_suppliers,
    "employees": df_employees_clean,
    "shippers": df_shippers,
}

for nom, df in dataframes_nettoyes.items():
    chemin = f"{TMP_PATH}/{nom}.parquet"

    (
        df.write
        .mode("overwrite")
        .parquet(chemin)
    )

    print(f"{nom} enregistré dans {chemin}")

In [ ]:
# Vérifier les fichiers parquet
for nom in dataframes_nettoyes:
    chemin = f"{TMP_PATH}/{nom}.parquet"

    df_verification = spark.read.parquet(chemin)

    print(
        nom,
        "=>",
        df_verification.count(),
        "ligne(s)"
    )

In [ ]:
df_customers_clean.withColumn(
        ,
        F.trim(F.col(colonne))
    )